# Neur — AS-RoPE vs RoPE vs Sinusoidal (Minimal A100 Pipeline)

**Goal:** Compare three positional encoding schemes on Hi→En NMT under a tight A100 budget (20–30h total).

**Design:**
- 1M deterministically-sampled pairs from the cleaned 5M Samanantar corpus
- Pre-tokenize once with IndicBART (Hi→En direction only), cache as flat int32 tensors
- 15K training steps per PE variant × 3 variants, effective batch = 256, bf16 + torch.compile + SDPA
- Fast greedy-only FLORES-200 eval (BLEU + chrF + TER) during the 3 runs
- One final full-metric pass (BLEU + chrF + TER + BERTScore + COMET + beam-5) on the winning PE variant

**Expected wallclock on A100:** ~3.5–4.5h total for all 3 runs + final eval. Leaves headroom for a second seed on the winner.

**Before you start:** Runtime → Change runtime type → GPU → **A100**.


## Step 0 — Mount Drive, clone repo, install deps

In [1]:
print("Hello")

Hello


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
%cd /content
!rm -rf /content/neur
!git clone https://github.com/thenileshmishra/AS-RoPE.git neur
%cd /content/neur
!git pull
!ls


/content
Cloning into 'neur'...
remote: Enumerating objects: 338, done.
remote: Counting objects: 100% (338/338), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 338 (delta 174), reused 271 (delta 107), pack-reused 0 (from 0)
Receiving objects: 100% (338/338), 1.10 MiB | 247.00 KiB/s, done.
Resolving deltas: 100% (174/174), done.
/content/neur
Already up to date.
notebooks  pipeline  README.md	requirements.txt  src


In [4]:
# Install minimal deps. Skips bert-score and unbabel-comet — those are only
# needed in step 4 (final eval on winner) and will be installed there.
!pip install -q transformers==4.44.2 sacrebleu tqdm sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.3 MB/s eta 0:00:00:00:01


In [5]:
import os, sys
os.environ['NEUR_DRIVE_ROOT'] = '/content/drive/MyDrive/neur'
sys.path.insert(0, '/content/neur')

from pipeline import paths
paths.ensure_dirs()
print(paths.summary())


PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics


In [6]:
# GPU sanity check — confirm A100 is attached
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'Memory: {props.total_memory / 1e9:.1f} GB')
    assert 'A100' in torch.cuda.get_device_name(0), 'Expected A100. Switch runtime type.'


CUDA: True
Device: Tesla T4
Memory: 15.6 GB


AssertionError: Expected A100. Switch runtime type.

## Step 1 — Download Samanantar + FLORES-200 (skip if already on Drive)

Skip this if you already have the raw data on Drive from a previous session.

In [ ]:
# Only run if you don't already have raw_data on Drive
!python -m pipeline.step1_download


## Step 2 — Clean and split (skip if already on Drive)

Skip this if `processed_data/train.tsv` already exists from a previous session.

In [ ]:
# Only run if processed_data is missing
!test -f /content/drive/MyDrive/neur/processed_data/train.tsv && echo 'processed_data exists, skipping' || python -m pipeline.step2_preprocess


In [ ]:
!ls -lh /content/drive/MyDrive/neur/processed_data/


## Step 2b — Sample 1M pairs + pre-tokenize (ONE-TIME, ~5–10 min)

Writes `train_1m_hi_en.pt` and `val_hi_en.pt` under `processed_data/tokenized/` on Drive. After this, training never touches the HuggingFace tokenizer.

In [ ]:
!python -m pipeline.step2b_sample_and_tokenize \
  --n-train 1000000 \
  --max-seq-len 170 \
  --seed 42


[step2b] loading train pairs from /content/drive/MyDrive/neur/processed_data/train.tsv


In [ ]:
# Inspect what we produced — pay attention to seq_len_p99; if much less than 192,
# you can lower --max-seq-len in step 3 for a free speedup.
!cat /content/drive/MyDrive/neur/processed_data/tokenized/tokenization_summary.json


## Step 2c — Copy tokenized files from Drive → local SSD

Drive I/O is 10–30× slower than Colab's local disk. Copy once, then train from `/content/`. This is the single biggest wall-clock win.

In [ ]:
!mkdir -p /content/tokenized
!cp /content/drive/MyDrive/neur/processed_data/tokenized/train_1m_hi_en.pt /content/tokenized/
!cp /content/drive/MyDrive/neur/processed_data/tokenized/val_hi_en.pt /content/tokenized/
!ls -lh /content/tokenized/


## Step 3a — Train SINUSOIDAL baseline (~50–65 min on A100)

Uses: bf16, torch.compile, SDPA, pre-tokenized cached dataset, 500-sample in-loop eval, tqdm progress. All three runs use identical hyperparameters; only `--pe-type` differs.

In [ ]:
!python -m pipeline.step3_train \
  --pe-type sinusoidal \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 15000 \
  --eval-every 3000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 170 \
  --seed 42


## Step 3b — Train ROPE (~50–65 min on A100)

In [ ]:
!python -m pipeline.step3_train \
  --pe-type rope \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 15000 \
  --eval-every 3000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 170 \
  --seed 42


## Step 3c — Train AS-ROPE (~50–65 min on A100)

In [ ]:
!python -m pipeline.step3_train \
  --pe-type asrope \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 15000 \
  --eval-every 3000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 170 \
  --seed 42


## Step 3d — Compare the three runs (fast metrics)

Reads the `metrics/<pe_type>/greedy/metrics.json` files and prints a side-by-side table.

In [ ]:
import json
from pathlib import Path

METRICS = Path('/content/drive/MyDrive/neur/outputs/metrics')
rows = []
for pe in ('sinusoidal', 'rope', 'asrope'):
    p = METRICS / pe / 'greedy' / 'metrics.json'
    if not p.exists():
        print(f'missing: {p}')
        continue
    m = json.loads(p.read_text())
    rows.append((pe, m.get('bleu', 0), m.get('chrf', 0), m.get('ter', 0)))

print(f"{'PE type':<14}{'BLEU':>10}{'chrF':>10}{'TER':>10}")
print('-' * 44)
for pe, bleu, chrf, ter in rows:
    print(f'{pe:<14}{bleu:>10.2f}{chrf:>10.2f}{ter:>10.2f}')

if rows:
    winner = max(rows, key=lambda r: r[1])
    print(f'\nwinner (by BLEU): {winner[0]}')


## Step 4 — Full metric suite on the winning PE (~15–25 min)

Run this ONCE on the winning checkpoint to get BERTScore + COMET + beam-5 for the paper's headline table. Edit the `--checkpoint` path to match your winner.

In [ ]:
# Install heavy metric deps only now — the 3 main runs skipped them
!pip install -q bert-score unbabel-comet


In [ ]:
# EDIT the --checkpoint arg to point at your winner (sinusoidal / rope / asrope)
WINNER = 'asrope'  # change this to whichever PE type won
!python -m pipeline.step4_final_eval \
  --checkpoint /content/drive/MyDrive/neur/outputs/checkpoints/$WINNER/best.pt \
  --run-name ${WINNER}_final


## Step 5 — (Optional) Second seed on the winner for significance

If you have A100 hours left after the 3 main runs and the final eval, rerun the winning PE type with `--seed 7` to establish that the gain isn't a single-seed artifact. This is the single most valuable addition for paper reviewers.

In [ ]:
# Only run if you have spare budget. Edit WINNER below.
WINNER = 'asrope'
!python -m pipeline.step3_train \
  --pe-type $WINNER \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 15000 \
  --eval-every 3000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 170 \
  --seed 7 \
  --run-name ${WINNER}_seed7


## Step 6 — Verify outputs on Drive

In [ ]:
!echo '--- checkpoints ---'
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints/*/
!echo
!echo '--- logs ---'
!ls -lh /content/drive/MyDrive/neur/outputs/logs/*/
!echo
!echo '--- metrics ---'
!find /content/drive/MyDrive/neur/outputs/metrics -name '*.json' | head -20
